In [1]:
import pandas as pd
#Step 1: Load the Dataset & Initial Inspection
df = pd.read_excel(r"C:\Users\HP\Downloads\foodtrends_facebook.xlsx")

# View first few rows
df.head()


,post_id,post_type,food_category,brand_name,post_date,post_time,likes_count,comments_count,shares_count,love,wow,sad,angry,day_of_week,time_of_day,total_engagement
0,1,image,burger,Subway,2025-01-01,00:00,100,28,17,13,5,1,3,Wednesday,NaN,167
1,2,video,sushi,Chipotle,2025-01-01,01:00,96,20,10,18,5,3,1,Wednesday,Morning,153
2,3,image,pizza,McDonalds,2025-01-01,02:00,102,20,10,21,6,3,1,Wednesday,Morning,163
3,4,image,pizza,Chipotle,2025-01-01,03:00,103,15,7,23,4,2,2,Wednesday,Morning,156
4,5,image,pizza,Chipotle,2025-01-01,04:00,110,23,6,17,6,1,0,Wednesday,Morning,163


In [2]:
# Step 2: Check & Handle Missing Values
df.isnull().sum()


post_id              0
post_type            0
food_category        0
brand_name           0
post_date            0
post_time            0
likes_count          0
comments_count       0
shares_count         0
love                 0
wow                  0
sad                  0
angry                0
day_of_week          0
time_of_day         42
total_engagement     0
dtype: int64

In [3]:
#Since time_of_day is derived from post_time, missing values were handled by recalculating the column instead of removing records.
# Convert post_time to hour if needed
df['post_time'] = pd.to_datetime(df['post_time'], format='%H:%M').dt.hour

# Function to assign time of day
def assign_time_of_day(hour):
    if 5 <= hour <= 11:
        return "Morning"
    elif 12 <= hour <= 16:
        return "Afternoon"
    elif 17 <= hour <= 20:
        return "Evening"
    else:
        return "Night"

# Recompute time_of_day
df['time_of_day'] = df['post_time'].apply(assign_time_of_day)

# Verify missing values again
df.isnull().sum()


post_id             0
post_type           0
food_category       0
brand_name          0
post_date           0
post_time           0
likes_count         0
comments_count      0
shares_count        0
love                0
wow                 0
sad                 0
angry               0
day_of_week         0
time_of_day         0
total_engagement    0
dtype: int64

In [4]:
#Step 3 Data Type Conversion & Standardization
df.dtypes


post_id                      int64
post_type                   object
food_category               object
brand_name                  object
post_date           datetime64[ns]
post_time                    int32
likes_count                  int64
comments_count               int64
shares_count                 int64
love                         int64
wow                          int64
sad                          int64
angry                        int64
day_of_week                 object
time_of_day                 object
total_engagement             int64
dtype: object

In [6]:
#Convert post_date to proper datetime format.
df['post_date'] = pd.to_datetime(df['post_date'])
#Standardize Text Columns
text_cols = ['post_type', 'food_category', 'brand_name', 'time_of_day']

for col in text_cols:
    df[col] = df[col].str.lower().str.strip()
#Ensure Numeric Columns Are Integers
num_cols = [
    'likes_count', 'comments_count', 'shares_count',
    'love', 'wow', 'sad', 'angry', 'total_engagement'
]

df[num_cols] = df[num_cols].astype(int)


In [7]:
#STEP 4: Duplicate Detection & Removal
# Check total duplicate rows
df.duplicated().sum()
# Check duplicate post_id values
df['post_id'].duplicated().sum()
df = df.drop_duplicates(subset='post_id')
#varify again 
df['post_id'].duplicated().sum()


np.int64(0)

In [9]:
# STEP 5: Recalculation & Validation of Total Engagement

# Recalculate total engagement from base metrics
df['total_engagement'] = (
    df['likes_count'] +
    df['comments_count'] +
    df['shares_count']
)

# Validate the calculation by viewing sample rows
df[['likes_count', 'comments_count', 'shares_count', 'total_engagement']].head()


,likes_count,comments_count,shares_count,total_engagement
0,100,28,17,145
1,96,20,10,126
2,102,20,10,132
3,103,15,7,125
4,110,23,6,139


In [10]:
#Step 6 Reaction & Sentiment Metrics Validation
#Check Basic Statistics of Reactions
df[['love', 'wow', 'sad', 'angry']].describe()
# Check if any reaction value is negative
(df[['love', 'wow', 'sad', 'angry']] < 0).sum()


love     0
wow      0
sad      0
angry    0
dtype: int64

In [11]:
#Trend & Time Feature Validation
#Recalculate day_of_week from post_date
df['day_of_week'] = df['post_date'].dt.day_name()
#we recheck 
df[['post_date', 'day_of_week']].head()
#We already recalculated time_of_day earlier, but now we confirm consistency.
df[['post_time', 'time_of_day']].head()


,post_time,time_of_day
0,0,night
1,1,night
2,2,night
3,3,night
4,4,night


In [12]:
#STEP 8: Final Dataset Export & Readiness Confirmation
# Final export of the preprocessed dataset
df.to_excel(r"C:\Users\HP\Downloads\foodtrends_facebook_cleaned.xlsx",index=False)



In [13]:
print("Final dataset exported successfully!")
print(df.shape)


Final dataset exported successfully!
(1000, 16)
